<a href="https://colab.research.google.com/github/jaroslav200720072007-beep/gradient-boosting-from-scratch/blob/main/Gradient_Boosting_on_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [142]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
import xgboost as xgb

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)
print(df.shape)
display(df.head())

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [143]:
if "customerID" in df.columns:
  df = df.drop(columns=["customerID"])

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)


target_col = "MonthlyCharges"
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

for col in cat_cols:
  unique_vals = df[col].unique()
  if len(unique_vals) == 2:
    df[col] = (df[col] == unique_vals[0]).astype(int)

df = pd.get_dummies(df, drop_first=True, dtype=int)

X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [144]:
df

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,1,0,1,1,1,1,1,29.85,29.85,1,...,0,0,0,0,0,0,0,0,1,0
1,0,0,0,1,34,0,0,56.95,1889.50,1,...,0,0,0,0,0,1,0,0,0,1
2,0,0,0,1,2,0,1,53.85,108.15,0,...,0,0,0,0,0,0,0,0,0,1
3,0,0,0,1,45,1,0,42.30,1840.75,1,...,1,0,0,0,0,1,0,0,0,0
4,1,0,0,1,2,0,1,70.70,151.65,0,...,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0,0,1,0,24,0,1,84.80,1990.50,1,...,1,0,1,0,1,1,0,0,0,1
7039,1,0,1,0,72,0,1,103.20,7362.90,1,...,0,0,1,0,1,1,0,1,0,0
7040,1,0,1,0,11,1,1,29.60,346.45,1,...,0,0,0,0,0,0,0,0,1,0
7041,0,1,1,1,4,0,1,74.40,306.60,0,...,0,0,0,0,0,0,0,0,0,1


In [145]:
class GradientBoostingHand:
  def __init__(self, gamma=0.0, n_estimators=100, learning_rate=0.05, random_state=10):
   self.n_estimators = n_estimators
   self.learning_rate = learning_rate
   self.random_state = random_state
   self.gamma = gamma

  def loss_gradient(self, y_true, y_pred):
    return y_true - y_pred


  def gamma_regul(self, tree):
     return self.gamma * tree.get_n_leaves()

  def fit(self, X, y):
    self.trees = []
    self.y_base = np.mean(y, axis=0)
    self.y_pred = self.y_base

    for i in range(self.n_estimators):
      gradient = self.loss_gradient(y, self.y_pred)
      clf = DecisionTreeRegressor(
          random_state=self.random_state, max_depth=4, min_samples_leaf=5)
      clf.fit(X, gradient)
      self.trees.append(clf)
      self.y_pred += (self.learning_rate * clf.predict(X))
      self.y_pred += self.learning_rate * clf.predict(X_train)

  def predict(self, X):
    y_pred = self.y_base + self.learning_rate * sum([tree.predict(X) for tree in self.trees])
    return y_pred

In [146]:
clf = GradientBoostingHand()
clf.fit(X_train, y_train)

In [147]:
y_pred = clf.predict(X_test)
y_pred

array([44.92441498, 44.92236034, 42.38400169, ..., 42.41171148,
       54.17625935, 47.52665505], shape=(1409,))

In [148]:
def calculate_rmse(y_true, y_pred):
  return np.sqrt(np.mean((y_true - y_pred) ** 2))

In [149]:
rmse_score = calculate_rmse(y_test, y_pred)
print("RMSE:", rmse_score)

RMSE: 15.088706060280717


In [150]:
def mean_squared_error(y_true, y_pred):
  return np.mean((y_true - y_pred) ** 2)

In [151]:
def overfiting(model, X_train, X_test, y_train, y_test):
  check_train = mean_squared_error(y_train, model.predict(X_train))
  check_test = mean_squared_error(y_test, model.predict(X_test))

  print(f'RMSE train = {np.sqrt(check_train)}')
  print(f'RMSE test = {np.sqrt(check_test)}')

  value = 1 - (check_test/check_train)

  return value

In [152]:
overfiting(clf, X_train, X_test, y_train, y_test)

RMSE train = 15.074719416865712
RMSE test = 15.088706060280717


np.float64(-0.001856503140097887)

In [153]:
##Here I want show how will be doing XGBoost this is separate library
import xgboost as xgb
xgb_regressor = xgb.XGBRegressor(random_state=42)
xgb_regressor.fit(X_train, y_train)
y_pred = xgb_regressor.predict(X_test)
print(f'RMSE: {calculate_rmse(y_test, y_pred)}')

RMSE: 1.1660668082730046


In [154]:
!pip install catboost
from catboost import CatBoostRegressor

cat_model = CatBoostRegressor(iterations=1000, learning_rate=0.03, depth=4, loss_function="RMSE", verbose=100)
cat_model.fit(X_train, y_train)
y_pred = cat_model.predict(X_test)
print(f'RMSE: {calculate_rmse(y_test, y_pred)}')

0:	learn: 29.2669324	total: 1.16ms	remaining: 1.16s
100:	learn: 3.0436817	total: 84.3ms	remaining: 750ms
200:	learn: 1.3061796	total: 167ms	remaining: 664ms
300:	learn: 1.0862974	total: 355ms	remaining: 824ms
400:	learn: 1.0274761	total: 649ms	remaining: 970ms
500:	learn: 1.0002024	total: 876ms	remaining: 872ms
600:	learn: 0.9833036	total: 992ms	remaining: 659ms
700:	learn: 0.9704568	total: 1.08s	remaining: 459ms
800:	learn: 0.9598319	total: 1.19s	remaining: 295ms
900:	learn: 0.9505252	total: 1.31s	remaining: 144ms
999:	learn: 0.9424142	total: 1.45s	remaining: 0us
RMSE: 1.041637958403858
